# Looming Analysis Notebook

This notebook provides tools for loading `.braidz` files and analyzing the angular velocity response of flies to looming stimuli.

In [ ]:
import zipfile
import gzip
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from typing import List, Optional
from scipy.signal import find_peaks
from scipy.stats import circmean
from pynumdiff.smooth_finite_difference import butterdiff

%matplotlib inline

## Helper Functions

We'll define functions to load data from `.braidz` zip archives and process trajectories.

In [ ]:
def load_braidz(file_path: str):
    """
    Load kalman estimates and stimulus data from a .braidz file.
    """
    with zipfile.ZipFile(file_path, "r") as z:
        # Load kalman_estimates.csv.gz
        with z.open("kalman_estimates.csv.gz") as f:
            with gzip.open(f) as gz:
                df_kalman = pl.read_csv(gz)

        # Load stim.csv or visual_stimuli.csv if it exists
        df_stim = None
        if "stim.csv" in z.namelist():
            with z.open("stim.csv") as f:
                df_stim = pl.read_csv(f)
        elif "visual_stimuli.csv" in z.namelist():
            with z.open("visual_stimuli.csv") as f:
                df_stim = pl.read_csv(f)

    return df_kalman, df_stim


def calculate_angular_velocity(
    xvel: np.ndarray, yvel: np.ndarray, dt: float = 0.01, params: list = [1, 0.1]
) -> np.ndarray:
    """
    Calculate angular velocity from velocity components.
    """
    theta = np.arctan2(yvel, xvel)
    theta_unwrap = np.unwrap(theta)
    # angular_velocity = np.gradient(theta_unwrap, dt)
    theta_unwrap, angular_velocity = butterdiff(theta_unwrap, dt=0.01, params=params)
    return angular_velocity


def extract_responses(
    df_kalman: pl.DataFrame,
    df_stim: pl.DataFrame,
    pre_frames: int = -50,
    post_frames: int = 100,
    k: int = 5,
):
    """
    Extract response trajectories for each stimulus.
    """

    if pre_frames > 0 or post_frames <= 0:
        raise ValueError(
            "pre_frames should be negative and post_frames should be positive."
        )

    responses = []
    dt = 0.01  # 100Hz

    # To make it efficient, we'll group kalman by obj_id
    kalman_grouped = df_kalman.partition_by("obj_id", as_dict=True)

    for row in df_stim.iter_rows(named=True):
        obj_id = row["obj_id"]
        stim_frame = row["frame"]

        obj_key = (obj_id,)
        if obj_key in kalman_grouped:
            df_obj = kalman_grouped[obj_key]

            # Create a target frame range
            target_frames = pl.DataFrame(
                {
                    "frame": np.arange(
                        stim_frame + pre_frames,
                        stim_frame + post_frames,
                        dtype=np.int64,
                    )
                }
            )

            # Join with object data to see what we have
            df_res = target_frames.join(df_obj, on="frame", how="left")

            # Check how many frames are missing
            missing_count = df_res["xvel"].null_count()

            if missing_count <= k:
                # Pad missing values (pre, post, or middle) using forward and backward fill
                df_res = df_res.fill_null(strategy="forward").fill_null(
                    strategy="backward"
                )

                # Calculate angular velocity
                xvel = df_res["xvel"].to_numpy()
                yvel = df_res["yvel"].to_numpy()

                # Calculate heading
                headings = np.arctan2(yvel, xvel)

                # Heading before: average during pre_frames
                stim_idx = abs(pre_frames)
                heading_before = circmean(
                    headings[stim_idx - 10 : stim_idx], low=-np.pi, high=np.pi
                )

                # Heading after: average during the period after expansion ends
                expansion_duration_ms = row.get("expansion_duration_ms", 500)
                expansion_frames = int(expansion_duration_ms / 10)
                end_expansion_idx = stim_idx + expansion_frames

                if end_expansion_idx < len(headings):
                    heading_after = circmean(
                        headings[end_expansion_idx : end_expansion_idx + 10],
                        low=-np.pi,
                        high=np.pi,
                    )
                else:
                    heading_after = headings[-1]

                # Heading change in degrees
                heading_change = np.rad2deg(
                    np.arctan2(
                        np.sin(heading_after - heading_before),
                        np.cos(heading_after - heading_before),
                    )
                )

                ang_vel = calculate_angular_velocity(xvel, yvel, dt, params=[2, 0.2])

                # Store response metadata
                response_data = {
                    "ang_vel": ang_vel,
                    "heading_change": heading_change,
                    "end_expansion_time": expansion_frames
                    * dt,  # seconds relative to stim onset
                    "time": (df_res["frame"].to_numpy() - stim_frame) * dt,
                    **{
                        key: val
                        for key, val in row.items()
                        if key not in ["frame", "timestamp", "obj_id"]
                    },
                }
                responses.append(response_data)

    return responses


def angular_velocity_from_velocity(
    xvel: np.ndarray,
    yvel: np.ndarray,
    dt: float,
    min_speed: float = 0.1,
    order: int = 2,
    cutoff: float = 0.2,
) -> np.ndarray:
    """Compute angular velocity directly from cartesian velocity components.

    Avoids arctan2 + unwrapping entirely. Numerically stable except near
    zero speed, where heading is undefined anyway.

    Args:
        xvel: x-component of velocity
        yvel: y-component of velocity
        dt: timestep
        min_speed: speed threshold below which output is set to NaN (heading undefined)
        order: order of the Butterworth filter
        cutoff: cutoff frequency of the Butterworth filter

    Returns:
        Angular velocity in radians/s
    """
    speed_sq = xvel**2 + yvel**2

    # Differentiate velocity components
    _, dxdt = butterdiff(xvel, dt, [order, cutoff])
    _, dydt = butterdiff(yvel, dt, [order, cutoff])

    omega = (xvel * dydt - yvel * dxdt) / speed_sq
    omega[speed_sq < min_speed**2] = np.nan

    return omega

## Data Loading and Processing

Let's load all `.braidz` files in the `data` directory and extract the responses.

In [ ]:
def process_all_files(
    file_paths: List[str],
    pre_frames: int = -50,
    post_frames: int = 100,
    group_name: Optional[str] = None,
):
    """
    Process multiple .braidz files and combine the responses.
    Optionally tag each response with a group name.
    """
    all_responses = []
    for path in file_paths:
        print(f"Processing {path}...")
        df_kalman, df_stim = load_braidz(path)
        if df_stim is not None:
            responses = extract_responses(
                df_kalman, df_stim, pre_frames=pre_frames, post_frames=post_frames
            )
            if group_name is not None:
                for r in responses:
                    r["group"] = group_name
            print(f"  Extracted {len(responses)} valid responses.")
            all_responses.extend(responses)
        else:
            print(f"  No stim data found in {path}.")
    return all_responses


def process_file_groups(
    file_groups: dict, pre_frames: int = -50, post_frames: int = 100
):
    """
    Process multiple groups of .braidz files.

    Args:
        file_groups: dict mapping group name -> list of file paths.
                     e.g. {"control": [...], "exp_1": [...], "exp_2": [...]}
    Returns:
        Combined list of responses, each tagged with a 'group' key.
    """
    all_responses = []
    for group_name, file_paths in file_groups.items():
        print(f"\n=== Group: {group_name} ===")
        responses = process_all_files(
            file_paths, pre_frames, post_frames, group_name=group_name
        )
        print(f"  Total for group '{group_name}': {len(responses)} responses.")
        all_responses.extend(responses)
    print(f"\nTotal responses across all groups: {len(all_responses)}")
    return all_responses


# --- Single-group usage (no group comparison) ---
# file_paths = [
#     "/mnt/data/experiments/20260407_171532.braidz",
#     "/mnt/data/experiments/20260410_143644.braidz",
# ]
# all_responses = process_all_files(file_paths, pre_frames=-50, post_frames=100)

# --- Multi-group usage ---
file_groups = {
    "CS": [
        "/mnt/data/experiments/20260415_154556.braidz",
    ],
    "DNp03": [
        "/mnt/data/experiments/20260410_143644.braidz",
        "/mnt/data/experiments/20260407_171532.braidz",
    ],
    # "experimental_2": [...],
}
all_responses = process_file_groups(file_groups, pre_frames=-50, post_frames=100)

## Grouping and Plotting

Now we'll group the responses by stimulus parameters and plot the average angular velocity response.

In [ ]:
def _prepare_ang_vel(response_list, time_axis, baseline_subtract):
    """Convert ang_vel to absolute deg/s, optionally subtracting per-trial baseline."""
    data_abs_deg = np.abs(np.rad2deg(np.stack([r["ang_vel"] for r in response_list])))
    if baseline_subtract:
        pre_mask = time_axis < 0
        baseline = np.nanmean(data_abs_deg[:, pre_mask], axis=1, keepdims=True)
        data_abs_deg = data_abs_deg - baseline
    return data_abs_deg


def plot_responses(
    responses,
    group_by: str = "stimulus_offset_deg",
    compare_by: Optional[str] = None,
    baseline_subtract: bool = True,
):
    """
    Plot mean angular velocity over time.

    Args:
        group_by:           Primary grouping variable (e.g. 'stimulus_offset_deg').
                            When compare_by is set, each value becomes a separate subplot.
        compare_by:         Secondary grouping variable for between-group comparison
                            (e.g. 'group'). Each value is drawn as a separate colored line.
                            If None, all responses are shown on a single axes.
        baseline_subtract:  If True (default), subtract each trial's mean |angular velocity|
                            during the pre-stimulus window before averaging. This removes the
                            offset introduced by taking the absolute value and anchors the
                            pre-stimulus baseline to ~0.
    """
    ylabel = (
        "Δ Angular velocity (deg/s)"
        if baseline_subtract
        else "Angular velocity (deg/s)"
    )

    if compare_by is None:
        groups = {}
        for r in responses:
            groups.setdefault(r.get(group_by), []).append(r)

        plt.figure(figsize=(10, 6))
        for label, response_list in groups.items():
            time_axis = response_list[0]["time"]
            data = _prepare_ang_vel(response_list, time_axis, baseline_subtract)
            mean_resp = np.nanmean(data, axis=0)
            sem_resp = np.nanstd(data, axis=0)
            plt.plot(
                time_axis,
                mean_resp,
                label=f"{group_by}={label} (n={len(response_list)})",
            )
            plt.fill_between(
                time_axis, mean_resp - sem_resp, mean_resp + sem_resp, alpha=0.2
            )

        plt.xlabel("Time relative to stimulus start (s)")
        plt.ylabel(ylabel)
        plt.title(f"Fly Response to Looming (grouped by {group_by})")
        plt.axvline(0, color="k", linestyle="--", alpha=0.5)
        plt.axhline(0, color="k", linestyle=":", alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    else:
        group_vals = sorted(
            {r.get(group_by) for r in responses if r.get(group_by) is not None}
        )
        compare_vals = sorted(
            {r.get(compare_by) for r in responses if r.get(compare_by) is not None}
        )

        n_subplots = len(group_vals)
        fig, axes = plt.subplots(
            1, n_subplots, figsize=(6 * n_subplots, 5), sharey=True
        )
        if n_subplots == 1:
            axes = [axes]

        colors = plt.cm.tab10(np.linspace(0, 0.9, len(compare_vals)))
        color_map = dict(zip(compare_vals, colors))

        for ax, gval in zip(axes, group_vals):
            for cval in compare_vals:
                subset = [
                    r
                    for r in responses
                    if r.get(group_by) == gval and r.get(compare_by) == cval
                ]
                if not subset:
                    continue
                time_axis = subset[0]["time"]
                data = _prepare_ang_vel(subset, time_axis, baseline_subtract)
                mean_resp = np.nanmean(data, axis=0)
                sem_resp = np.nanstd(data, axis=0)
                ax.plot(
                    time_axis,
                    mean_resp,
                    color=color_map[cval],
                    label=f"{cval} (n={len(subset)})",
                )
                ax.fill_between(
                    time_axis,
                    mean_resp - sem_resp,
                    mean_resp + sem_resp,
                    alpha=0.2,
                    color=color_map[cval],
                )

            ax.axvline(0, color="k", linestyle="--", alpha=0.5)
            ax.axhline(0, color="k", linestyle=":", alpha=0.3)
            ax.set_title(f"{group_by} = {gval}")
            ax.set_xlabel("Time relative to stimulus start (s)")
            ax.grid(True, alpha=0.3)

        axes[0].set_ylabel(ylabel)
        axes[-1].legend(title=compare_by, bbox_to_anchor=(1.05, 1), loc="upper left")
        fig.suptitle(
            f"Fly Response to Looming\ngrouped by {group_by}, compared by {compare_by}"
        )
        plt.tight_layout()
        plt.show()


def plot_heading_changes(
    responses, group_by: str = "stimulus_offset_deg", compare_by: Optional[str] = None
):
    """
    Plot the distribution of heading changes.

    Args:
        group_by:   Primary grouping variable (x-axis ticks).
        compare_by: Secondary grouping variable for between-group comparison.
                    Each value is shown as a differently-colored violin within
                    each group_by tick. If None, all responses are shown together.
    """
    if compare_by is None:
        groups = {}
        for r in responses:
            groups.setdefault(r.get(group_by), []).append(r["heading_change"])

        labels = sorted(groups.keys())
        data = [groups[label] for label in labels]

        plt.figure(figsize=(10, 6))
        plt.violinplot(data, showmeans=True)
        plt.xticks(range(1, len(labels) + 1), labels)
        plt.xlabel(group_by)
        plt.ylabel("Heading change (deg)")
        plt.title(f"Heading Change Distribution (grouped by {group_by})")
        plt.axhline(0, color="k", linestyle="--", alpha=0.3)
        plt.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    else:
        from matplotlib.patches import Patch

        group_vals = sorted(
            {r.get(group_by) for r in responses if r.get(group_by) is not None}
        )
        compare_vals = sorted(
            {r.get(compare_by) for r in responses if r.get(compare_by) is not None}
        )

        n_compare = len(compare_vals)
        colors = plt.cm.tab10(np.linspace(0, 0.9, n_compare))
        color_map = dict(zip(compare_vals, colors))

        fig, ax = plt.subplots(figsize=(max(8, 2.5 * len(group_vals) * n_compare), 6))

        tick_positions = []
        tick_labels = []
        x = 0
        violin_width = 0.7
        gap = 1.0

        for gval in group_vals:
            group_start = x
            for cval in compare_vals:
                subset = [
                    r["heading_change"]
                    for r in responses
                    if r.get(group_by) == gval and r.get(compare_by) == cval
                ]
                if subset:
                    vp = ax.violinplot(
                        [subset], positions=[x], widths=violin_width, showmeans=True
                    )
                    for pc in vp["bodies"]:
                        pc.set_facecolor(color_map[cval])
                        pc.set_alpha(0.7)
                    for partname in ("cbars", "cmins", "cmaxes", "cmeans"):
                        if partname in vp:
                            vp[partname].set_color(color_map[cval])
                x += 1

            tick_positions.append((group_start + x - 1) / 2)
            tick_labels.append(str(gval))
            x += gap

        legend_handles = [
            Patch(facecolor=color_map[cv], alpha=0.7, label=str(cv))
            for cv in compare_vals
        ]
        ax.legend(
            handles=legend_handles,
            title=compare_by,
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
        )
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels)
        ax.set_xlabel(group_by)
        ax.set_ylabel("Heading change (deg)")
        ax.set_title(
            f"Heading Change Distribution\ngrouped by {group_by}, compared by {compare_by}"
        )
        ax.axhline(0, color="k", linestyle="--", alpha=0.3)
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()


if all_responses:
    # Single-group plots
    # plot_responses(all_responses, group_by='stimulus_offset_deg')
    # plot_heading_changes(all_responses, group_by='stimulus_offset_deg')

    # Multi-group comparison
    if any("group" in r for r in all_responses):
        plot_responses(
            all_responses, group_by="stimulus_offset_deg", compare_by="group"
        )
        plot_heading_changes(
            all_responses, group_by="stimulus_offset_deg", compare_by="group"
        )

In [ ]:
def classify_responsiveness(responses, threshold_deg_s: float = 300.0):
    """
    Tag each response dict with responsiveness metadata (in-place).

    A trial is 'responsive' if a peak detected by scipy.find_peaks in the
    |angular velocity| trace falls within ±window_ms of end_expansion_time
    AND reaches or exceeds threshold_deg_s.

    Adds to each response:
        'is_responsive'      bool
        'peak_ang_vel_deg_s' float  (highest qualifying peak, NaN if none found)
    """
    for r in responses:
        # time = r['time']
        # ang_vel_abs_deg = np.abs(np.rad2deg(r['ang_vel']))
        # end_t = r['end_expansion_time']

        # # Replace NaNs with 0 for peak finding (find_peaks can't handle NaNs)
        # trace = np.where(np.isnan(ang_vel_abs_deg), 0.0, ang_vel_abs_deg)

        # peak_indices, _ = find_peaks(trace, height=threshold_deg_s, distance=10)

        # # Keep only peaks inside the time window
        # window_mask = np.abs(time[peak_indices] - end_t) <= half_win
        # peaks_in_window = peak_indices[window_mask]

        # if peaks_in_window.size > 0:
        #     peak = float(np.max(ang_vel_abs_deg[peaks_in_window]))
        # else:
        #     peak = np.nan

        # r['peak_ang_vel_deg_s'] = peak
        # r['is_responsive'] = (not np.isnan(peak)) and (peak >= threshold_deg_s)
        ang_vel_abs_deg = np.abs(np.rad2deg(r["ang_vel"]))
        peak_indices, _ = find_peaks(ang_vel_abs_deg, height=threshold_deg_s)

        peaks_in_window = np.array([pi for pi in peak_indices if 50 <= pi <= 100])

        if peaks_in_window.size > 0:
            peak_index = peaks_in_window[0]
            peak_value = ang_vel_abs_deg[peak_index]
        else:
            peak_index = np.nan
            peak_value = np.nan

        r["peak_ang_vel_deg_s"] = peak_value
        r["is_responsive"] = not np.isnan(peak_value)

    return responses


def plot_responsiveness_rates(
    responses, group_by: str = "stimulus_offset_deg", compare_by: Optional[str] = None
):
    """
    Bar chart of the percentage of responsive trials per condition (and group).

    Each bar is annotated with the total n for that cell.
    """
    group_vals = sorted(
        {r.get(group_by) for r in responses if r.get(group_by) is not None}
    )

    if compare_by is None:
        rates, ns = [], []
        for gval in group_vals:
            subset = [r for r in responses if r.get(group_by) == gval]
            n_resp = sum(r["is_responsive"] for r in subset)
            rates.append(100 * n_resp / len(subset) if subset else 0)
            ns.append(len(subset))

        fig, ax = plt.subplots(figsize=(max(6, len(group_vals) * 1.2), 5))
        bars = ax.bar(range(len(group_vals)), rates, color="steelblue", width=0.6)
        for bar, n in zip(bars, ns):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1,
                f"n={n}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
        ax.set_xticks(range(len(group_vals)))
        ax.set_xticklabels([str(v) for v in group_vals])
        ax.set_xlabel(group_by)
        ax.set_ylabel("Responsive trials (%)")
        ax.set_title(f"Response Rate by {group_by}")
        ax.set_ylim(0, 110)
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    else:
        compare_vals = sorted(
            {r.get(compare_by) for r in responses if r.get(compare_by) is not None}
        )
        n_compare = len(compare_vals)
        colors = plt.cm.tab10(np.linspace(0, 0.9, n_compare))
        color_map = dict(zip(compare_vals, colors))

        bar_width = 0.8 / n_compare
        fig, ax = plt.subplots(figsize=(max(8, 2 * len(group_vals) * n_compare), 5))

        for i, cval in enumerate(compare_vals):
            rates, ns = [], []
            for gval in group_vals:
                subset = [
                    r
                    for r in responses
                    if r.get(group_by) == gval and r.get(compare_by) == cval
                ]
                n_resp = sum(r["is_responsive"] for r in subset)
                rates.append(100 * n_resp / len(subset) if subset else 0)
                ns.append(len(subset))

            x = (
                np.arange(len(group_vals))
                + i * bar_width
                - (n_compare - 1) * bar_width / 2
            )
            bars = ax.bar(
                x,
                rates,
                width=bar_width,
                color=color_map[cval],
                label=str(cval),
                alpha=0.85,
                edgecolor="white",
            )
            for bar, n in zip(bars, ns):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 1,
                    f"n={n}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

        ax.set_xticks(range(len(group_vals)))
        ax.set_xticklabels([str(v) for v in group_vals])
        ax.set_xlabel(group_by)
        ax.set_ylabel("Responsive trials (%)")
        ax.set_title(f"Response Rate by {group_by}, compared by {compare_by}")
        ax.set_ylim(0, 115)
        ax.legend(title=compare_by, bbox_to_anchor=(1.05, 1), loc="upper left")
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()


def plot_responses_by_responsiveness(
    responses,
    group_by: str = "stimulus_offset_deg",
    compare_by: Optional[str] = None,
    baseline_subtract: bool = True,
):
    """
    2-row grid: top = responsive trials, bottom = non-responsive trials.
    Columns = one per group_by value.  Lines colored by compare_by (if given).

    A vertical dashed line marks stimulus onset (t=0).
    """
    group_vals = sorted(
        {r.get(group_by) for r in responses if r.get(group_by) is not None}
    )
    n_cols = len(group_vals)

    if compare_by is not None:
        compare_vals = sorted(
            {r.get(compare_by) for r in responses if r.get(compare_by) is not None}
        )
        colors = plt.cm.tab10(np.linspace(0, 0.9, len(compare_vals)))
        color_map = dict(zip(compare_vals, colors))
    else:
        compare_vals = [None]
        color_map = {None: "steelblue"}

    ylabel = (
        "Δ Angular velocity (deg/s)"
        if baseline_subtract
        else "Angular velocity (deg/s)"
    )

    fig, axes = plt.subplots(
        2, n_cols, figsize=(5 * n_cols, 8), sharey="row", sharex=True
    )
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    row_labels = ["Responsive", "Non-responsive"]

    for row_idx, is_resp in enumerate([True, False]):
        for col_idx, gval in enumerate(group_vals):
            ax = axes[row_idx, col_idx]

            for cval in compare_vals:
                if compare_by is None:
                    subset = [
                        r
                        for r in responses
                        if r.get(group_by) == gval and r.get("is_responsive") == is_resp
                    ]
                else:
                    subset = [
                        r
                        for r in responses
                        if r.get(group_by) == gval
                        and r.get(compare_by) == cval
                        and r.get("is_responsive") == is_resp
                    ]

                if not subset:
                    continue

                time_axis = subset[0]["time"]
                data = _prepare_ang_vel(subset, time_axis, baseline_subtract)
                mean_resp = np.nanmean(data, axis=0)
                sem_resp = np.nanstd(data, axis=0)
                label = (
                    f"{cval} (n={len(subset)})" if compare_by else f"n={len(subset)}"
                )
                ax.plot(time_axis, mean_resp, color=color_map[cval], label=label)
                ax.fill_between(
                    time_axis,
                    mean_resp - sem_resp,
                    mean_resp + sem_resp,
                    alpha=0.2,
                    color=color_map[cval],
                )

            ax.axvline(0, color="k", linestyle="--", alpha=0.5)
            ax.axhline(0, color="k", linestyle=":", alpha=0.3)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8, loc="upper left")

            if row_idx == 0:
                ax.set_title(f"{group_by} = {gval}")
            if col_idx == 0:
                ax.set_ylabel(f"{row_labels[row_idx]}\n{ylabel}")
            if row_idx == 1:
                ax.set_xlabel("Time relative to stimulus start (s)")

    title = f"Angular velocity: Responsive vs Non-responsive  |  grouped by {group_by}"
    if compare_by:
        title += f", compared by {compare_by}"
    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
classify_responsiveness(all_responses, threshold_deg_s=1000)

n_resp = sum(r["is_responsive"] for r in all_responses)
print(
    f"Responsive: {n_resp} / {len(all_responses)} ({100 * n_resp / len(all_responses):.1f}%)"
)

has_groups = any("group" in r for r in all_responses)

if all_responses:
    # (a) Response rates per condition
    plot_responsiveness_rates(all_responses, group_by="stimulus_offset_deg")
    if has_groups:
        plot_responsiveness_rates(
            all_responses, group_by="stimulus_offset_deg", compare_by="group"
        )

    # (b) Angular velocity traces split by responsiveness
    plot_responses_by_responsiveness(
        all_responses,
        group_by="stimulus_offset_deg",
        compare_by="group" if has_groups else None,
    )